# Multi-Norm Adversarial Training

Use Google Colab with a GPU: **Runtime > Change runtime type > T4 GPU**.

All the code is in this notebook. Complete each place that has a `TODO` and `raise NotImplementedError`.
Do not change the data, the model, the attacks, the evaluation or the default hyperparameters,
unless the homework tells you to change them.

**Threat model.** For each test image $x$, the perturbation region is
$B_\infty(x, 8/255) \cup B_2(x, 0.5) \cup B_1(x, 12)$.
The union accuracy is the percentage of test images that are robust against all three attacks simultaneously.

# Set up for dataset and model

Download the base model, a subset of 10,000 CIFAR-10 training images (1,000 for each class),
and the first 1,000 CIFAR-10 test images from Google Drive.

In [1]:
!pip install -q gdown

In [2]:
!gdown 17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes -O linf_base.pt
!gdown 1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe -O cifar10_train_10k.pt
!gdown 1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA -O cifar10_test_1k.pt

Downloading...
From: https://drive.google.com/uc?id=17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes
To: /kaggle/working/linf_base.pt
100%|██████████████████████████████████████| 11.2M/11.2M [00:00<00:00, 91.4MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe
From (redirected): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe&confirm=t&uuid=9f12e8d9-6ec2-4826-95c2-a272fa7fd6ce
To: /kaggle/working/cifar10_train_10k.pt
100%|███████████████████████████████████████| 30.8M/30.8M [00:00<00:00, 102MB/s]
Downloading...
From: https://drive.google.com/uc?id=1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA
To: /kaggle/working/cifar10_test_1k.pt
100%|██████████████████████████████████████| 3.08M/3.08M [00:00<00:00, 87.3MB/s]


In [3]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

NORMS = ("linf", "l2", "l1")
EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}

device: cuda


In [4]:
def load_data(path):
    # The files store images as uint8. Convert them to floats in [0, 1] on the GPU.
    x, y = torch.load(path)
    return x.float().div(255).to(device), y.to(device)


# Training data: 10,000 images. Test data: the first 1,000 CIFAR-10 test images.
x_train, y_train = load_data("cifar10_train_10k.pt")
x_test, y_test = load_data("cifar10_test_1k.pt")
print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape))


def augment(x):
    # Random horizontal flip and random crop (padding 4) for each image.
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)
    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]

train: (10000, 3, 32, 32) test: (1000, 3, 32, 32)


In [5]:
# PreActResNet-18 with half width (2.8M parameters). The input is an image in [0, 1].
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)


def load_base_model():
    model = make_model()
    model.load_state_dict(torch.load("linf_base.pt", map_location=device))
    return model

# Implement the Attacks

Each attack is untargeted, uses the cross-entropy loss, and returns `x_adv` with
$\|x_{adv} - x\|_p \leq \epsilon$ and $x_{adv} \in [0, 1]$. The attacks must not change the mode (`train` or `eval`) of the model.

* `pgd_linf`: start at a uniformly random point in the $\ell_\infty$ ball. For each step, add `alpha * sign(grad)`,
  clip the perturbation to $[-\epsilon, \epsilon]$, and clip the image to $[0, 1]$.
* `pgd_l2`: start at a random point in the $\ell_2$ ball. For each step, add `alpha * grad / ||grad||_2`,
  scale the perturbation back to the $\ell_2$ ball if its norm is larger than $\epsilon$, and clip the image to $[0, 1]$.

You can use your code from HW1. The $\ell_1$ attack (sparse $\ell_1$ descent, Tramer & Boneh 2019) is complete.

In [6]:
def clip_box(x, delta):
    # Make x + delta an image in [0, 1]. This never increases |delta_i|.
    return (x + delta).clamp(0, 1) - x


def input_grad(model, x, delta, y):
    delta = delta.detach().requires_grad_(True)
    loss = F.cross_entropy(model(x + delta), y, reduction="sum")
    (grad,) = torch.autograd.grad(loss, delta)
    return grad.detach()


def l2_norm(v):
    return v.flatten(1).norm(dim=1).view(-1, 1, 1, 1)


# PGD attacks based on HW1
def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):
    ce_loss = torch.nn.CrossEntropyLoss()
    adv_x = x.clone().detach()
    if rand_init:
        adv_x = torch.clamp(x + torch.empty_like(x).uniform_(-eps, eps), 0, 1).detach()
    for _ in range(steps):
        adv_x.requires_grad_(True)
        model.zero_grad()
        output = model(adv_x)
        loss = ce_loss(output, y)
        loss.backward()
        gradient = adv_x.grad.data
        adv_x = adv_x + alpha * torch.sign(gradient)
        delta = adv_x - x
        delta = torch.clamp(delta, -eps, eps)
        adv_x = torch.clamp(x + delta, 0, 1).detach()
    return adv_x


def pgd_l2(model, x, y, eps, alpha, steps, rand_init=True):
    ce_loss = torch.nn.CrossEntropyLoss()
    batch_size = x.size()[0]
    adv_x = x.clone().detach()
    if rand_init:
        d = torch.randn_like(x)
        r = torch.rand(batch_size, 1, 1, 1, device=x.device)
        d = d / torch.norm(d.view(batch_size, -1), dim=1).view(batch_size, 1, 1, 1) * eps * r
        adv_x = torch.clamp(x + d, 0, 1).detach()
    for _ in range(steps):
        adv_x.requires_grad_(True)
        model.zero_grad()
        output = model(adv_x)
        loss = ce_loss(output, y)
        loss.backward()
        grad = adv_x.grad.data
        grad_norm = torch.norm(grad.view(batch_size, -1), dim=1).view(batch_size, 1, 1, 1)
        adv_x = adv_x.detach() + alpha * grad / (grad_norm + 1e-10)
        delta = adv_x - x
        delta_norm = torch.norm(delta.view(batch_size, -1), dim=1).view(batch_size, 1, 1, 1)
        delta = eps * delta / torch.clamp(delta_norm, eps)
        adv_x = torch.clamp(x + delta, 0, 1).detach()
    return adv_x

In [7]:
def project_l1(delta, eps):
    # Projection of each image of delta onto the L1 ball of radius eps (Duchi et al., 2008).
    v = delta.flatten(1)
    a = v.abs()
    outside = a.sum(dim=1) > eps
    if not outside.any():
        return delta
    mu, _ = a.sort(dim=1, descending=True)
    cumsum = mu.cumsum(dim=1)
    idx = torch.arange(1, v.shape[1] + 1, device=v.device, dtype=v.dtype)
    rho = ((mu - (cumsum - eps) / idx > 0) * idx).argmax(dim=1, keepdim=True)
    theta = (cumsum.gather(1, rho) - eps) / (rho + 1).to(v.dtype)
    w = (a - theta).clamp_min(0) * v.sign()
    return torch.where(outside.unsqueeze(1), w, v).view(delta.shape)


def pgd_l1(model, x, y, eps, alpha, steps, rand_init=True, k=20):
    # Each step changes only the k coordinates with the largest gradient.
    # Coordinates at the image boundary that point outward are skipped.
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.distributions.Laplace(0.0, 1.0).sample(x.shape).to(x.device)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / delta.flatten(1).abs().sum(1).view(-1, 1, 1, 1) * eps * r
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y).flatten(1)
        x_cur = (x + delta).flatten(1)
        blocked = ((grad > 0) & (x_cur >= 1)) | ((grad < 0) & (x_cur <= 0))
        grad = grad.masked_fill(blocked, 0)
        top = grad.abs().topk(k, dim=1).indices
        step = torch.zeros_like(grad).scatter(1, top, grad.gather(1, top).sign())
        delta = clip_box(x, delta + alpha * (step / k).view_as(x))
        delta = project_l1(delta, eps)
    return (x + delta).detach()


ATTACKS = {"linf": pgd_linf, "l2": pgd_l2, "l1": pgd_l1}


def attack(model, x, y, norm, steps=10):
    # The attack for one norm with the default eps and step size.
    alpha = EPS["l1"] / steps ** 0.5 if norm == "l1" else 2.5 * EPS[norm] / steps
    return ATTACKS[norm](model, x, y, EPS[norm], alpha, steps)

# Evaluate Single and Multi-Norm Robust Accuracy

`test_model` gives the clean accuracy, the robust accuracy for each norm, and the union accuracy (in %) on the
1,000 test images. An image is robust for a norm if the model is correct on the clean image and on the result
of each restart of the attack. An image is robust for the union if it is robust for all three norms simultaneously.
Use the default settings (`steps=50, restarts=2`) for all the numbers in your report.

In [8]:
def test_model(model, steps=50, restarts=2, batch_size=250):
    model.eval()
    torch.manual_seed(0)  # the same random starts of the attacks for each model, so the results are comparable
    start = time.time()
    correct = {k: 0 for k in ("clean", *NORMS, "union")}
    for i in range(0, len(x_test), batch_size):
        x, y = x_test[i:i + batch_size], y_test[i:i + batch_size]
        with torch.no_grad():
            clean = model(x).argmax(1) == y
        union = clean.clone()
        for norm in NORMS:
            robust = clean.clone()
            for _ in range(restarts):
                x_adv = attack(model, x, y, norm, steps)
                with torch.no_grad():
                    robust &= model(x_adv).argmax(1) == y
            correct[norm] += robust.sum().item()
            union &= robust
        correct["clean"] += clean.sum().item()
        correct["union"] += union.sum().item()
    acc = {k: 100.0 * v / len(y_test) for k, v in correct.items()}
    print("  ".join(f"{k} {v:.1f}" for k, v in acc.items()) + f"   ({time.time() - start:.0f} s)")
    return acc

In [9]:
# The base model: L-infinity PGD adversarial training for 30 epochs on all 50,000 training images.
test_model(load_base_model())

clean 78.7  linf 48.4  l2 59.8  l1 22.3  union 22.2   (115 s)


{'clean': 78.7, 'linf': 48.4, 'l2': 59.8, 'l1': 22.3, 'union': 22.2}

# Adversarial Training

`adv_train(model, loss_fn, ...)` fine-tunes `model` for `epochs` epochs on the training subset.
For each batch, it calls `loss_fn(model, x, y, **kwargs)` and does one optimizer step on the loss.

A loss function makes the adversarial examples itself. Put the model in `eval` mode for the attack
(so that the attack does not change the BatchNorm statistics), and in `train` mode before you calculate the loss.

If `gp_beta` is set, `adv_train` does the gradient projection of RAMP (part (b)) before the optimizer step.

In [10]:
def adv_train(model, loss_fn, epochs=10, lr=0.01, batch_size=128, gp_beta=None, seed=0, **kwargs):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))
    params = list(model.parameters())
    start = time.time()
    for epoch in range(epochs):
        losses = []
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:  # skip the last, smaller batch
                continue
            x = augment(x)
            loss = loss_fn(model, x, y, **kwargs)
            opt.zero_grad()
            if gp_beta is None:
                loss.backward()
            else:
                # RAMP (part (b)): combine the adversarial gradient g_a and the natural gradient g_n
                # for each parameter tensor. gradient_projection is defined in part (b).
                g_a = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, ga, gn in zip(params, g_a, g_n):
                    p.grad = gp_beta * gradient_projection(gn, ga) + (1 - gp_beta) * ga
            opt.step()
            sched.step()
            losses.append(loss.item())
        print(f"epoch {epoch + 1}: loss {np.mean(losses):.3f} ({time.time() - start:.0f} s)")
    model.eval()
    return model

In [11]:
def train_and_test(loss_fn, seed=0, **kwargs):
    start = time.time()
    model = adv_train(load_base_model(), loss_fn, seed=seed, **kwargs)
    time_per_epoch = (time.time() - start) / 10   # adv_train trains for 10 epochs
    print("Training time per epoch (s):", round(time_per_epoch, 1))
    acc = test_model(model)
    return model, acc, time_per_epoch

# Part (a): Single-Norm Adversarial Training

Complete `single_norm_loss`: attack the batch with the norm `norm` (use `attack`), and return the
cross-entropy loss of the model on the adversarial images.

In [12]:
def single_norm_loss(model, x, y, norm):
    model.eval()
    x_adv = attack(model, x, y, norm)
    model.train()
    return F.cross_entropy(model(x_adv), y)

In [13]:
model_linf, acc_linf, time_linf = train_and_test(single_norm_loss, norm="linf")

epoch 1: loss 1.203 (47 s)
epoch 2: loss 1.180 (92 s)
epoch 3: loss 1.151 (138 s)
epoch 4: loss 1.126 (184 s)
epoch 5: loss 1.097 (230 s)
epoch 6: loss 1.068 (276 s)
epoch 7: loss 1.049 (322 s)
epoch 8: loss 1.030 (368 s)
epoch 9: loss 1.013 (414 s)
epoch 10: loss 1.008 (460 s)
Training time per epoch (s): 46.3
clean 78.5  linf 44.7  l2 56.9  l1 20.8  union 20.7   (117 s)


In [14]:
model_l2, acc_l2, time_l2 = train_and_test(single_norm_loss, norm="l2")

epoch 1: loss 0.897 (46 s)
epoch 2: loss 0.826 (92 s)
epoch 3: loss 0.787 (139 s)
epoch 4: loss 0.754 (185 s)
epoch 5: loss 0.714 (231 s)
epoch 6: loss 0.675 (277 s)
epoch 7: loss 0.631 (323 s)
epoch 8: loss 0.610 (369 s)
epoch 9: loss 0.591 (416 s)
epoch 10: loss 0.583 (462 s)
Training time per epoch (s): 46.2
clean 82.6  linf 32.3  l2 60.0  l1 27.7  union 24.4   (117 s)


In [15]:
model_l1, acc_l1, time_l1 = train_and_test(single_norm_loss, norm="l1")

epoch 1: loss 1.231 (25 s)
epoch 2: loss 0.896 (50 s)
epoch 3: loss 0.617 (75 s)
epoch 4: loss 0.500 (100 s)
epoch 5: loss 0.364 (125 s)
epoch 6: loss 0.264 (150 s)
epoch 7: loss 0.216 (175 s)
epoch 8: loss 0.175 (200 s)
epoch 9: loss 0.147 (226 s)
epoch 10: loss 0.150 (251 s)
Training time per epoch (s): 25.1
clean 83.9  linf 0.0  l2 0.2  l1 24.9  union 0.0   (117 s)


# Part (b): Multi-Norm Adversarial Training (RAMP)

* `ramp_loss`: attack with $q = \ell_\infty$ and $r = \ell_1$. The loss is the mean of
  $\max(\mathcal{L}_{CE}(x^q), \mathcal{L}_{CE}(x^r))$ plus `lam` times $\mathcal{L}_{KL} = KL(p_q \| p_r)$.
  Calculate $\mathcal{L}_{KL}$ only on the images that the model classifies correctly under the $\ell_q$ attack.
* `gradient_projection(g_n, g_a)`: $\cos(g_n, g_a) \cdot g_n$ if $\cos(g_n, g_a) > 0$, else $0$.

`adv_train(..., gp_beta=0.5)` then uses `p.grad = gp_beta * GP(g_n, g_a) + (1 - gp_beta) * g_a` for each parameter
tensor, where $g_a$ is the gradient of `ramp_loss` and $g_n$ is the gradient of the cross-entropy loss on the clean images.

In [ ]:
def ramp_loss(model, x, y, lam=0.5):
    model.eval()
    x_q = attack(model, x, y, "linf")
    x_r = attack(model, x, y, "l1")
    model.train()

    logits_q = model(x_q)
    logits_r = model(x_r)
    ce_q = F.cross_entropy(logits_q, y, reduction="none")
    ce_r = F.cross_entropy(logits_r, y, reduction="none")
    loss = torch.max(ce_q, ce_r).mean()

    correct = logits_q.argmax(dim=1) == y
    if correct.sum() > 0:
        p_q = F.softmax(logits_q[correct], dim=1)
        log_p_q = F.log_softmax(logits_q[correct], dim=1)
        log_p_r = F.log_softmax(logits_r[correct], dim=1)
        kl = (p_q * (log_p_q - log_p_r)).sum(dim=1).mean()   # 1/n_c * sum over the correct images
        loss = loss + lam * kl
    return loss


def gradient_projection(g_n, g_a):
    cos = (g_n * g_a).sum() / (g_n.norm() * g_a.norm() + 1e-12)
    if cos > 0:
        return cos * g_n
    return 0

In [17]:
model_ramp_seed0, acc_ramp_seed0, time_ramp_seed0 = train_and_test(ramp_loss, seed=0, gp_beta=0.5)

epoch 1: loss 1.431 (76 s)
epoch 2: loss 1.330 (151 s)
epoch 3: loss 1.303 (227 s)
epoch 4: loss 1.281 (302 s)
epoch 5: loss 1.254 (377 s)
epoch 6: loss 1.228 (453 s)
epoch 7: loss 1.199 (529 s)
epoch 8: loss 1.186 (604 s)
epoch 9: loss 1.171 (680 s)
epoch 10: loss 1.166 (755 s)
Training time per epoch (s): 75.5
clean 79.2  linf 41.2  l2 61.6  l1 44.8  union 39.1   (117 s)


# Part (c): Your Method

Write the training loss of your method in `my_loss`, and train with `adv_train(load_base_model(), my_loss)`.
If your method needs a different update (not the gradient of one loss), write your own copy of `adv_train`.
Run your method and RAMP with `seed=0` and `seed=1`.
Then evaluate your best model again with stronger attacks: `test_model(model, steps=200, restarts=5)`.

## My method: training on L1 while keeping the base model's L-infinity robustness

The base model is already robust to L-infinity attacks. But in part (a), when I fine-tuned it
only on L1, it lost almost all of its L-infinity robustness.
So my idea is to train on L1 attacks as usual, but at the same time make the model keep giving
the same answers as the base model on L-infinity attacks.

For each batch, the loss is calculated as the sum of:
1. Cross-entropy on the L1 adversarial images. This teaches the model L1 robustness.
2. The KL divergence between the predictions of the base model and my model on the
   L-infinity adversarial images. This stops the model from forgetting L-infinity.

In [18]:
frozen_base = load_base_model()
frozen_base.eval()
for p in frozen_base.parameters():
    p.requires_grad_(False)

ce_log = []
kl_log = []

def my_loss(model, x, y, linf_weight):
    model.eval()
    x_l1 = attack(model, x, y, "l1")
    x_linf = attack(model, x, y, "linf")
    model.train()

    ce = F.cross_entropy(model(x_l1), y)

    log_p_model = F.log_softmax(model(x_linf), dim=1)
    with torch.no_grad():
        base_logits = frozen_base(x_linf)
        p_base = F.softmax(base_logits, dim=1)
        log_p_base = F.log_softmax(base_logits, dim=1)
    kl = (p_base * (log_p_base - log_p_model)).sum(dim=1).mean()

    ce_log.append(ce.item())
    kl_log.append(kl.item())
    return ce + linf_weight * kl


def my_loss_true_labels(model, x, y, linf_weight):
    # Ablation 1
    model.eval()
    x_l1 = attack(model, x, y, "l1")
    x_linf = attack(model, x, y, "linf")
    model.train()

    ce = F.cross_entropy(model(x_l1), y)
    ce_linf = F.cross_entropy(model(x_linf), y)
    return ce + linf_weight * ce_linf


def my_loss_base_correct(model, x, y, linf_weight):
    # Ablation 2
    model.eval()
    x_l1 = attack(model, x, y, "l1")
    x_linf = attack(model, x, y, "linf")
    model.train()

    ce = F.cross_entropy(model(x_l1), y)

    log_p_model = F.log_softmax(model(x_linf), dim=1)
    with torch.no_grad():
        base_logits = frozen_base(x_linf)
        p_base = F.softmax(base_logits, dim=1)
        log_p_base = F.log_softmax(base_logits, dim=1)
        base_correct = base_logits.argmax(dim=1) == y

    if base_correct.sum() > 0:
        kl_each_image = (p_base * (log_p_base - log_p_model)).sum(dim=1)
        kl = kl_each_image[base_correct].mean()
        return ce + linf_weight * kl
    else:
        return ce


def print_loss_log():
    n = len(ce_log) // 10
    for e in range(10):
        ce = np.mean(ce_log[e * n:(e + 1) * n])
        kl = np.mean(kl_log[e * n:(e + 1) * n])
        print("Epoch", e + 1, "Cross-entropy on L1:", round(ce, 3), "KL to base model:", round(kl, 3))

In [19]:
# Choosing lambda on validation images

lambda_values = [1.0, 3.0, 5.0, 7.0, 10.0, 30.0]

x_train_full, y_train_full = x_train, y_train
x_test_real, y_test_real = x_test, y_test

torch.manual_seed(0)
perm = torch.randperm(len(x_train_full), device=x_train_full.device)
val_idx = perm[:1000]
train_idx = perm[1000:]

validation_results = []


x_train, y_train = x_train_full[train_idx], y_train_full[train_idx]
x_test, y_test = x_train_full[val_idx], y_train_full[val_idx]

for lam in lambda_values:
    print("lambda =", lam)
    ce_log.clear()
    kl_log.clear()
    model_val, acc_val, time_val = train_and_test(my_loss, seed=0, linf_weight=lam)
    n = max(1, len(ce_log) // 10)   # batches in one epoch
    last_ce = np.mean(ce_log[-n:])
    last_kl = np.mean(kl_log[-n:])
    validation_results.append((lam, acc_val, last_ce, last_kl))

x_train, y_train = x_train_full, y_train_full
x_test, y_test = x_test_real, y_test_real

# lambda with the highest union accuracy
best_lambda = lambda_values[0]
best_union = -1.0
print("Validation results (accuracy in %, loss parts in the last epoch):")
for lam, acc, last_ce, last_kl in validation_results:
    print("lambda =", lam, " clean:", round(acc["clean"], 1), " linf:", round(acc["linf"], 1),
          " l2:", round(acc["l2"], 1), " l1:", round(acc["l1"], 1), " union:", round(acc["union"], 1),
          " CE on L1:", round(last_ce, 3), " KL to base:", round(last_kl, 3))
    if acc["union"] > best_union:
        best_union = acc["union"]
        best_lambda = lam
print("Best lambda on the validation images:", best_lambda, "(union", round(best_union, 1), ")")

lambda = 1.0
epoch 1: loss 1.357 (64 s)
epoch 2: loss 1.175 (129 s)
epoch 3: loss 1.095 (194 s)
epoch 4: loss 1.022 (259 s)
epoch 5: loss 0.981 (323 s)
epoch 6: loss 0.933 (387 s)
epoch 7: loss 0.885 (452 s)
epoch 8: loss 0.866 (517 s)
epoch 9: loss 0.839 (581 s)
epoch 10: loss 0.840 (646 s)
Training time per epoch (s): 64.6
clean 81.6  linf 39.4  l2 64.3  l1 56.4  union 38.8   (117 s)
lambda = 3.0
epoch 1: loss 1.537 (64 s)
epoch 2: loss 1.385 (129 s)
epoch 3: loss 1.300 (193 s)
epoch 4: loss 1.216 (258 s)
epoch 5: loss 1.162 (322 s)
epoch 6: loss 1.093 (386 s)
epoch 7: loss 1.041 (451 s)
epoch 8: loss 1.019 (516 s)
epoch 9: loss 0.984 (580 s)
epoch 10: loss 0.989 (645 s)
Training time per epoch (s): 64.5
clean 80.6  linf 43.4  l2 65.2  l1 54.4  union 42.4   (117 s)
lambda = 5.0
epoch 1: loss 1.756 (64 s)
epoch 2: loss 1.639 (129 s)
epoch 3: loss 1.506 (193 s)
epoch 4: loss 1.381 (258 s)
epoch 5: loss 1.310 (323 s)
epoch 6: loss 1.211 (387 s)
epoch 7: loss 1.146 (452 s)
epoch 8: loss 

In [20]:
# RAMP with seed 1
model_ramp_seed1, acc_ramp_seed1, time_ramp_seed1 = train_and_test(ramp_loss, seed=1, gp_beta=0.5)

# My method with the chosen lambda, seed 0 and seed 1
print("My method with lambda =", best_lambda)
ce_log.clear()
kl_log.clear()
model_my_method_seed0, acc_my_method_seed0, time_my_method_seed0 = train_and_test(my_loss, seed=0, linf_weight=best_lambda)
print_loss_log()

ce_log.clear()
kl_log.clear()
model_my_method_seed1, acc_my_method_seed1, time_my_method_seed1 = train_and_test(my_loss, seed=1, linf_weight=best_lambda)

print("Mean of seed 0 and seed 1:")
for key in ["clean", "linf", "l2", "l1", "union"]:
    print(key, " RAMP:", round((acc_ramp_seed0[key] + acc_ramp_seed1[key]) / 2, 1),
          " My method:", round((acc_my_method_seed0[key] + acc_my_method_seed1[key]) / 2, 1))
print("Training time per epoch (s)  RAMP:", round((time_ramp_seed0 + time_ramp_seed1) / 2, 1),
      " My method:", round((time_my_method_seed0 + time_my_method_seed1) / 2, 1))

epoch 1: loss 1.434 (76 s)
epoch 2: loss 1.333 (151 s)
epoch 3: loss 1.299 (227 s)
epoch 4: loss 1.274 (302 s)
epoch 5: loss 1.248 (378 s)
epoch 6: loss 1.221 (454 s)
epoch 7: loss 1.200 (530 s)
epoch 8: loss 1.186 (605 s)
epoch 9: loss 1.178 (681 s)
epoch 10: loss 1.168 (757 s)
Training time per epoch (s): 75.7
clean 79.0  linf 42.4  l2 62.6  l1 46.6  union 40.7   (117 s)
My method with lambda = 5.0
epoch 1: loss 1.751 (72 s)
epoch 2: loss 1.646 (144 s)
epoch 3: loss 1.546 (216 s)
epoch 4: loss 1.437 (287 s)
epoch 5: loss 1.326 (359 s)
epoch 6: loss 1.225 (431 s)
epoch 7: loss 1.150 (503 s)
epoch 8: loss 1.120 (575 s)
epoch 9: loss 1.095 (647 s)
epoch 10: loss 1.089 (718 s)
Training time per epoch (s): 71.8
clean 78.3  linf 42.7  l2 61.9  l1 50.4  union 40.9   (116 s)
Epoch 1 Cross-entropy on L1: 1.326 KL to base model: 0.085
Epoch 2 Cross-entropy on L1: 1.216 KL to base model: 0.086
Epoch 3 Cross-entropy on L1: 1.162 KL to base model: 0.077
Epoch 4 Cross-entropy on L1: 1.11 KL to bas

In [21]:
# Ablations
model_true_labels, acc_true_labels, time_true_labels = train_and_test(my_loss_true_labels, seed=0, linf_weight=best_lambda)

model_base_correct, acc_base_correct, time_base_correct = train_and_test(my_loss_base_correct, seed=0, linf_weight=best_lambda)

epoch 1: loss 8.875 (71 s)
epoch 2: loss 8.569 (142 s)
epoch 3: loss 8.260 (212 s)
epoch 4: loss 7.970 (284 s)
epoch 5: loss 7.608 (355 s)
epoch 6: loss 7.216 (426 s)
epoch 7: loss 6.821 (497 s)
epoch 8: loss 6.471 (567 s)
epoch 9: loss 6.198 (638 s)
epoch 10: loss 6.073 (709 s)
Training time per epoch (s): 70.9
clean 76.8  linf 43.5  l2 59.6  l1 40.3  union 37.6   (116 s)
epoch 1: loss 2.141 (72 s)
epoch 2: loss 2.056 (144 s)
epoch 3: loss 1.834 (215 s)
epoch 4: loss 1.666 (287 s)
epoch 5: loss 1.487 (359 s)
epoch 6: loss 1.323 (431 s)
epoch 7: loss 1.215 (503 s)
epoch 8: loss 1.164 (575 s)
epoch 9: loss 1.138 (647 s)
epoch 10: loss 1.126 (719 s)
Training time per epoch (s): 71.9
clean 78.1  linf 41.3  l2 61.7  l1 49.2  union 39.6   (117 s)


In [22]:
#Stronger Attacks
if acc_my_method_seed0["union"] >= acc_my_method_seed1["union"]:
    my_method_best = model_my_method_seed0
    my_method_regular = acc_my_method_seed0
    print("My method: using seed 0")
else:
    my_method_best = model_my_method_seed1
    my_method_regular = acc_my_method_seed1
    print("My method: using seed 1")

if acc_ramp_seed0["union"] >= acc_ramp_seed1["union"]:
    ramp_best = model_ramp_seed0
    ramp_regular = acc_ramp_seed0
    print("RAMP: using seed 0")
else:
    ramp_best = model_ramp_seed1
    ramp_regular = acc_ramp_seed1
    print("RAMP: using seed 1")

print("My method, stronger evaluation (200 steps, 5 restarts):")
my_method_stronger = test_model(my_method_best, steps=200, restarts=5)
print("RAMP, stronger evaluation (200 steps, 5 restarts):")
ramp_stronger = test_model(ramp_best, steps=200, restarts=5)

print("Union accuracy, regular evaluation (50 steps, 2 restarts) -> stronger evaluation (200 steps, 5 restarts):")
print("My method:", my_method_regular["union"], "->", my_method_stronger["union"])
print("RAMP: ", ramp_regular["union"], "->", ramp_stronger["union"])

My method: using seed 1
RAMP: using seed 1
My method, stronger evaluation (200 steps, 5 restarts):
clean 78.7  linf 42.4  l2 61.6  l1 48.3  union 40.9   (1164 s)
RAMP, stronger evaluation (200 steps, 5 restarts):
clean 79.0  linf 42.0  l2 62.4  l1 45.1  union 39.7   (1164 s)
Union accuracy, regular evaluation (50 steps, 2 restarts) -> stronger evaluation (200 steps, 5 restarts):
My method: 41.3 -> 40.9
RAMP:  40.7 -> 39.7
